# Promo Finder
Reads a microdrama's timestamped screenplays and returns a ranked **promo cut sheet**: for each clip, the episode, the start and end of every segment, the hook, the cliffhanger, why it works, and the posting order.

**Run all cells** (Runtime → Run all). By default it runs on a small built-in test show, so you can see it work without any confidential scripts. To use your own show, set `USE_DEMO_SHOW = False` in step 2 and upload a `.zip` of the episode `.txt` files.

The tool works without any API key (rules only). To add the AI judge, set `PROVIDER` in step 2 and paste a key when asked. The key stays in this session only.

## 1 · Load the tool

In [ ]:
import base64, io, zipfile, sys, os, subprocess
PKG = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG))).extractall("/content/promo_finder_app" if os.path.isdir("/content") else "promo_finder_app")
APP = "/content/promo_finder_app" if os.path.isdir("/content") else os.path.abspath("promo_finder_app")
subprocess.run([sys.executable, "tests/make_synthetic_show.py"], cwd=APP, check=True, capture_output=True)
print("Tool loaded:", sorted(os.listdir(os.path.join(APP, "promo_finder"))))

## 2 · Settings

In [ ]:
USE_DEMO_SHOW = True        # False = upload your own .zip of episode .txt files
STRATEGY = "funnel"         # "funnel" = premise episodes first · "peak" = strongest moments first
PROVIDER = "heuristic"      # "heuristic" (no key) · "gemini" · "openai" (any OpenAI-compatible API) · "anthropic"
MODEL = ""                  # blank = default for the provider, e.g. gemini-3.5-flash-lite
OPENAI_BASE_URL = ""        # only for openai-compatible endpoints other than OpenAI (Groq, OpenRouter, Kuku's own API)

## 3 · Choose the show

In [ ]:
INPUT = os.path.join(APP, "tests/synthetic_show/The Waitress Was His Wife")
if not USE_DEMO_SHOW:
    try:
        from google.colab import files
        up = files.upload()               # pick the .zip of screenplays
        INPUT = os.path.abspath(next(iter(up)))
    except ImportError:
        INPUT = input("Path to the show's .zip or folder: ")
print("Input:", INPUT)

## 4 · Find the promos

In [ ]:
env = dict(os.environ)
KEYS = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
if PROVIDER in KEYS and not env.get(KEYS[PROVIDER]):
    from getpass import getpass
    env[KEYS[PROVIDER]] = getpass(f"{KEYS[PROVIDER]}: ")
if OPENAI_BASE_URL:
    env["OPENAI_BASE_URL"] = OPENAI_BASE_URL
OUT = os.path.join(APP, "out")
cmd = [sys.executable, "-m", "promo_finder", "run", INPUT, "--out", OUT, "--provider", PROVIDER, "--strategy", STRATEGY]
if MODEL:
    cmd += ["--model", MODEL]
r = subprocess.run(cmd, cwd=APP, env=env, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])

## 5 · The cut sheet
Accept or reject each promo; **Export my decisions** downloads a CSV that `python -m promo_finder.learn` uses to re-fit the ranking.

In [ ]:
from IPython.display import HTML, display
display(HTML(open(os.path.join(OUT, "cut_sheet.html")).read()))

## 6 · Checks

In [ ]:
r = subprocess.run([sys.executable, "-m", "promo_finder.evaluate", OUT, INPUT], cwd=APP, capture_output=True, text=True)
print(r.stdout[:2500])
if USE_DEMO_SHOW:   # the test show has an answer key written before the tool was run on it
    r = subprocess.run([sys.executable, "tests/score_planted.py", OUT], cwd=APP, capture_output=True, text=True)
    print(r.stdout[-700:])

## 7 · Download all outputs

In [ ]:
import shutil
z = shutil.make_archive(os.path.join(APP, "promo_outputs"), "zip", OUT)
try:
    from google.colab import files; files.download(z)
except ImportError:
    print("Saved:", z)